# 05 — The recovery grid

Forty full-posterior fits over a factorial design: two panel lengths, two
collinearity levels, two specifications, five seeds. This notebook fits nothing;
it reads what `scripts/run_recovery.py` produced.

Both length arms use the *simulated* extended baseline, including the 85-week arm,
so that length is the only quantity changing along the length axis. Only the
headline fit in notebook 03 uses the real Olist baseline.

In [1]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [2]:
recovery = read_metric("recovery", METRICS)
print(json.dumps(recovery["design"], indent=2))
print()
print(json.dumps(recovery["convergence"], indent=2))

{
  "baseline": "Both length arms use the simulated extended baseline, including the 85-week arm, so that length is the only quantity changing along the length axis. Only the headline fit uses the real Olist baseline.",
  "cells": 40,
  "chains": 4,
  "collinearity_levels": [
    "low",
    "high"
  ],
  "draws": 1000,
  "hdi_prob": 0.89,
  "panel_lengths": [
    85,
    156
  ],
  "priors": "pymc-marketing defaults, unchanged. A prior centred near the true ROI would produce excellent recovery and prove nothing.",
  "seeds": [
    20260829,
    20260830,
    20260831,
    20260901,
    20260902
  ],
  "specifications": {
    "matched": "DelayedAdstock + HillSaturation \u2014 the generating form, as a control",
    "misspecified": "GeometricAdstock + LogisticSaturation \u2014 cannot express the generating form"
  },
  "tune": 1000
}

{
  "cells_that_errored": [],
  "converged": 21,
  "converged_under_strict_rule": 15,
  "criteria": "divergence rate < 0.5% of post-warmup draws, max R-hat

## Coverage, and why coverage alone is not enough

Coverage is the property a Bayesian model claims: the truth should fall inside the
89% interval about 89% of the time. But an interval can achieve perfect coverage by
being uselessly wide, so interval width is reported beside it. A cell with coverage
1.00 and a median error of twenty is not a success.

In [3]:
rows = []
for name, slice_ in recovery["slices"].items():
    for quantity in ("average_roi", "marginal_roi"):
        block = slice_[quantity]
        if block.get("converged", 0) == 0:
            rows.append({"cell": name, "quantity": quantity, "converged": 0})
            continue
        rows.append({
            "cell": name, "quantity": quantity,
            "converged": block["converged"],
            "coverage": block["coverage_rate"],
            "median |rel err|": round(block["median_absolute_relative_error"], 3),
            "mean interval width": round(block["mean_interval_width"], 3),
        })
show(pd.DataFrame(rows).sort_values(["quantity", "cell"]), "Every cell of the grid")

Every cell of the grid
                  cell     quantity  converged  coverage  median |rel err|  mean interval width
     156w_high_matched  average_roi          0       NaN               NaN                  NaN
156w_high_misspecified  average_roi          3    0.4000             0.593                3.940
      156w_low_matched  average_roi          3    0.9333             8.801               56.253
 156w_low_misspecified  average_roi          5    0.6400             0.837               10.614
      85w_high_matched  average_roi          0       NaN               NaN                  NaN
 85w_high_misspecified  average_roi          4    0.8500             0.718                7.822
       85w_low_matched  average_roi          3    1.0000             7.579               40.331
  85w_low_misspecified  average_roi          3    0.7333             0.870                8.666
     156w_high_matched marginal_roi          0       NaN               NaN                  NaN
156w_high_misspec

## Does the choice of convergence rule change the conclusion

The primary rule uses a divergence rate; the strict rule demands zero divergences
and rejected almost every fit when it was first applied. Both are computed for
every cell so the effect of the choice is visible rather than asserted.

In [4]:
rows = []
for name, slice_ in recovery["slices"].items():
    primary = slice_["average_roi"]
    strict = slice_.get("average_roi_strict_convergence", {})
    rows.append({
        "cell": name,
        "converged (rate rule)": primary.get("converged", 0),
        "coverage (rate rule)": primary.get("coverage_rate"),
        "converged (strict)": strict.get("converged", 0),
        "coverage (strict)": strict.get("coverage_rate"),
    })
show(pd.DataFrame(rows))
print(recovery["convergence"]["rule_note"])

                  cell  converged (rate rule)  coverage (rate rule)  converged (strict)  coverage (strict)
     156w_high_matched                      0                   NaN                   0                NaN
156w_high_misspecified                      3                0.4000                   3             0.4000
      156w_low_matched                      3                0.9333                   0                NaN
 156w_low_misspecified                      5                0.6400                   5             0.6400
      85w_high_matched                      0                   NaN                   0                NaN
 85w_high_misspecified                      4                0.8500                   4             0.8500
       85w_low_matched                      3                1.0000                   0                NaN
  85w_low_misspecified                      3                0.7333                   3             0.7333

The primary rule uses a divergence r

## Error against the identification diagnostics

If recovery error tracks the condition number and the media signal share, then the
failures are a property of the *design* rather than of the method — which is the
more useful thing for a practitioner to know, because a design is something they
control.

In [5]:
fits = pd.DataFrame([{
    "weeks": f["weeks"], "collinearity": f["collinearity"], "specification": f["specification"],
    "converged": f["diagnostics"]["passed"],
    "condition_number": f["identification"]["condition_number"],
    "media_signal": f["identification"]["media_share_of_detrended_variance"],
    "coverage": f["average_roi"]["summary"]["coverage_rate"],
    "median_abs_rel_error": f["average_roi"]["summary"]["median_absolute_relative_error"],
    "mean_interval_width": f["average_roi"]["summary"]["mean_interval_width"],
} for f in recovery["fits"]])
usable = fits[fits["converged"]]
show(usable.groupby(["specification", "collinearity", "weeks"]).agg(
    fits=("coverage", "size"),
    coverage=("coverage", "mean"),
    median_error=("median_abs_rel_error", "median"),
    interval_width=("mean_interval_width", "mean"),
    condition=("condition_number", "mean"),
    signal=("media_signal", "mean"),
).round(3).reset_index())

specification collinearity  weeks  fits  coverage  median_error  interval_width  condition  signal
      matched          low     85     3     1.000         7.579          40.331      1.510   0.041
      matched          low    156     3     0.933         8.801          56.253      1.423   0.029
 misspecified         high     85     4     0.850         0.718           7.822      5.263   0.085
 misspecified         high    156     3     0.400         0.593           3.940      5.678   0.076
 misspecified          low     85     3     0.733         0.870           8.666      1.510   0.041
 misspecified          low    156     5     0.640         0.837          10.614      1.403   0.034



## Per channel

Which channels are recoverable is not uniform, and the pattern is worth more than
the average: a channel that is never recovered is a channel a media-mix model
should not be used to budget.

In [6]:
rows = []
for name, slice_ in recovery["slices"].items():
    block = slice_["average_roi"]
    for channel, entry in block.get("per_channel", {}).items():
        rows.append({"cell": name, "channel": channel, **entry})
if rows:
    per_channel = pd.DataFrame(rows)
    show(per_channel.groupby("channel").agg(
        true=("true", "first"),
        median_estimate=("median_estimate", "median"),
        coverage=("coverage_rate", "mean"),
        median_rel_error=("median_relative_error", "median"),
    ).round(3).reset_index(), "Across every converged cell")

Across every converged cell
        channel  true  median_estimate  coverage  median_rel_error
   display_prog   0.9            5.043     0.878             4.603
   search_brand   1.6           10.364     0.822             5.477
search_nonbrand   2.8            2.102     0.703            -0.249
    social_paid   2.1            2.041     0.781            -0.028
      video_ctv   2.5            7.822     0.614             2.129

